In [1]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles()
styles.register_and_enable_theme(theme_name='article')

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [2]:
xmax=9
mo=0.5
logo=alt.Chart(pd.DataFrame([{"x": xmax, "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40,height=40,align='right',baseline='top',yOffset=-33,opacity=mo,xOffset=0).encode(x='x:Q',url='img:N')

logo

alt.Chart(...)

### USA fuel prices

Datasource: [EIA](https://www.eia.gov/petroleum/gasdiesel/)

- `Data 1` Diesel: Weekly U.S. No 2 Diesel Retail Prices  (Dollars per Gallon) - `EMD_EPD2D_PTE_NUS_DPG`
- `Data 3` Gasoline: Weekly U.S. Regular All Formulations Retail Gasoline Prices  (Dollars per Gallon) `EMM_EPMR_PTE_NUS_DPG`

In [160]:
# download latest data and save .xls files

import requests

url = 'https://www.eia.gov/petroleum/gasdiesel/xls/pswrgvwall.xls'
response = requests.get(url)
with open('eia-gas-history-pswrgvwall.xls', 'wb') as file:
    file.write(response.content)


url = 'https://www.eia.gov/petroleum/gasdiesel/xls/psw18vwall.xls'
response = requests.get(url)
with open('eia-diesel-history-psw18vwall.xls', 'wb') as file:
    file.write(response.content)

In [83]:
diesel = pd.read_excel('eia-diesel-history-psw18vwall.xls', sheet_name='Data 1', skiprows=2)

# Remove 'No 2 Diesel Retail Prices  (Dollars per Gallon)' from column names
diesel.columns = diesel.columns.str.replace(' No 2 Diesel Retail Prices  (Dollars per Gallon)', '')
diesel.columns = diesel.columns.str.replace('Weekly ', '')
diesel = diesel.rename(columns={'Date': 'date', 'U.S.': 'USA'})

gas = pd.read_excel('eia-gas-history-pswrgvwall.xls', sheet_name='Data 3', skiprows=2)
gas.columns = gas.columns.str.replace(' Regular All Formulations Retail Gasoline Prices  (Dollars per Gallon)', '')
gas.columns = gas.columns.str.replace('Weekly ', '')
gas = gas.rename(columns={'Date': 'date', 'U.S.': 'USA'})


gas['type'] = 'Gasoline'
diesel['type'] = 'Diesel'


diesel

,date,USA,East Coast,New England (PADD 1A),Central Atlantic (PADD 1B),Lower Atlantic (PADD 1C),Midwest,Gulf Coast,Rocky Mountain,West Coast,California,West Coast (PADD 5) Except California,type
0,1994-03-21,1.106,1.119,NaN,NaN,NaN,1.087,1.065,1.105,1.209,NaN,NaN,Diesel
1,1994-03-28,1.107,1.115,NaN,NaN,NaN,1.089,1.064,1.100,1.220,NaN,NaN,Diesel
2,1994-04-04,1.109,1.122,NaN,NaN,NaN,1.087,1.069,1.103,1.219,NaN,NaN,Diesel
3,1994-04-11,1.108,1.119,NaN,NaN,NaN,1.090,1.066,1.111,1.209,NaN,NaN,Diesel
4,1994-04-18,1.105,1.115,NaN,NaN,NaN,1.086,1.058,1.118,1.213,NaN,NaN,Diesel
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1666,2026-02-23,3.809,3.843,4.201,4.104,3.708,3.798,3.489,3.683,4.465,4.944,4.050,Diesel
1667,2026-03-02,3.897,3.924,4.314,4.122,3.808,3.888,3.598,3.737,4.534,4.990,4.138,Diesel
1668,2026-03-09,4.859,4.901,4.970,4.940,4.880,4.801,4.627,4.397,5.556,6.096,5.088,Diesel
1669,2026-03-16,5.071,5.105,5.236,5.196,5.057,4.970,4.835,4.796,5.856,6.428,5.360,Diesel


### Chart 1. Long-run, USA wide

- ECO green: #00A767
- Prep dat

In [58]:
styles.eco_colours

{'red': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4'}

Prep data

In [59]:
usa_long = pd.concat([gas[['date', 'USA', 'type']], diesel[['date', 'USA', 'type']]])

In [60]:
base = alt.Chart(usa_long).transform_filter(
    "year(datum.date) >= 1995"
).encode(
    alt.X('date:T').axis(tickCount=8),
    alt.Y('USA:Q').scale(zero=False, domain=[0.3,6.1], nice=False).axis(
        labelExpr="'$' + datum.value",
        labelFontSize=13, domain=False
    ).title('Dollars per gallon'),
    alt.Color('type:N').legend(None).scale(range=['#00A767', '#122b39'])
)

lines = base.mark_line(interpolate='basis', strokeWidth=1.5)

labels = base.mark_text(
    align='left',
    dx=5,
    dy=0
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('USA:Q').aggregate({'argmax': 'date'}),
    alt.Text('type:N')
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')

chart = alt.layer(lines, labels, logo)
styles.add_source(chart, 'Source: EIA; regular all formulations gasoline & No.2 diesel, weekly retail prices')
styles.save(chart, 'charts', f"usa-fuel-prices-1995", width=350, height=260)
chart.display()

alt.LayerChart(...)

**Since 2016**

In [55]:
usa_long = pd.read_csv('usa_average_petrol_prices.csv')

In [ ]:
if not usa_long:
    usa_long = pd.read_csv('usa_average_petrol_prices.csv')
else:
    usa_long.to_csv('usa_average_petrol_prices.csv', index=False)


NameError: name 'usa_long' is not defined

In [61]:
start_year = 2015
base = alt.Chart(usa_long).transform_filter(
    f"year(datum.date) >= {start_year}"
).encode(
    alt.X('date:T').axis(),
    alt.Y('USA:Q').scale(zero=False, domain=[0.7, 6.1], nice=False).axis(
        labelExpr="'$' + datum.value",
        labelFontSize=13, domain=False, tickCount=7,
        titleFontSize=12
    ).title('Dollars per gallon'),
    alt.Color('type:N').legend(None).scale(range=['#00A767', '#122b39'])
)

lines = base.mark_line(interpolate='basis')

labels = base.mark_text(
    align='left',
    dx=5,
    dy=0
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('USA:Q').aggregate({'argmax': 'date'}),
    alt.Text('type:N')
)

rule1 = alt.Chart(pd.DataFrame({'date': ['2022-02-24'], 'label': ['Russia invades |Ukraine']})).mark_rule(strokeWidth=1.5, opacity=0.5).encode(
    alt.X('date:T')
)
label1 = rule1.mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), y=alt.value(15)
)
rule2 = alt.Chart(pd.DataFrame({'start': ['2026-02-28'], 'end': [diesel['date'].max()], 'label': ['US & Israel |strikes on Iran']})).mark_rect(opacity=0.9).encode(
    alt.X('start:T'), alt.X2('end:T')
)

label2 = alt.Chart(pd.DataFrame({'date': ['2026-02-28'], 'label': ['US & Israel |strikes on Iran']})).mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), alt.X('date:T'), y=alt.value(15)
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')

chart = rule1 + label1 + rule2 + label2 + alt.layer(lines, labels, logo)
# chart = rule1 + label1 + rule2 + label2 + styles.add_shaded_area(start_date='2026-02-28', end_date=petrol['date'].max()) + alt.layer(lines, labels)
chart = styles.add_source(chart, 'Source: EIA; regular all formulations gasoline & No.2 diesel, weekly retail prices')

styles.save(chart, 'charts', f"usa-fuel-prices-{start_year}", width=350, height=260)
chart.display()

alt.LayerChart(...)

In [ ]:
usa_long[usa_long['date'] >= '2026-02-20']

**Since 2024**

In [78]:
usa_long

,date,USA,type
0,1990-08-20,1.191,Gasoline
1,1990-08-27,1.245,Gasoline
2,1990-09-03,1.242,Gasoline
3,1990-09-10,1.252,Gasoline
4,1990-09-17,1.266,Gasoline
...,...,...,...
1666,2026-02-23,3.809,Diesel
1667,2026-03-02,3.897,Diesel
1668,2026-03-09,4.859,Diesel
1669,2026-03-16,5.071,Diesel


In [81]:
usa_long['date'] = usa_long['date'].dt.strftime('%Y-%m-%d')

In [ ]:
usa

In [85]:
base = alt.Chart(usa_long).transform_filter(
    "datum.date >= toDate('2024-12-25')"
).encode(
    alt.X('date:T').axis(
        tickCount=9,
        labelExpr="month(datum.value) == 0 ? year(datum.value) : monthAbbrevFormat(month(datum.value))"
    ),
    alt.Y('USA:Q').scale(zero=False, domain=[0.5, 6], nice=False).axis(
        labelExpr="'$' + datum.value",
        labelFontSize=13, domain=False, tickCount=5,
        titleFontSize=12
    ).title('Dollars per gallon'),
    alt.Color('type:N').legend(None).scale(range=['#00A767', '#122b39'])
)

lines = base.mark_line(interpolate='basis')

end_labels = base.mark_text(
    align='left',
    dx=5,
    dy=0
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('USA:Q').aggregate({'argmax': 'date'}),
    alt.Text('type:N')
)

# chart = alt.layer(lines, labels)

labels = alt.Chart(pd.DataFrame({'date': ['2025-06-13'], 'label': ['US/Israel & Iran | twelve-day war']})).mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), alt.X('date:T'), y=alt.value(15)
)

labels += alt.Chart(pd.DataFrame({'date': ['2026-02-29'], 'label': ['US/Israel |strikes on Iran']})).mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), alt.X('date:T'), y=alt.value(15)
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')


# Add shaded areas
chart = labels + styles.add_shaded_area(start_date='2025-06-13', end_date='2025-06-24') + styles.add_shaded_area(start_date='2026-02-28', end_date=gas['date'].max()) + alt.layer(lines, end_labels, logo)
chart = styles.add_source(chart, 'Source: EIA (24 March); regular all formulations gasoline & No.2 diesel, weekly retail prices')
styles.save(chart, 'charts', f"usa-fuel-prices-2025", width=350, height=260)
chart.display()

alt.LayerChart(...)

In [172]:
usa_long[usa_long['date'] >= '2026-02-20']

,date,USA,type
1853,2026-02-23,2.937,Gasoline
1854,2026-03-02,3.015,Gasoline
1855,2026-03-09,3.502,Gasoline
1856,2026-03-16,3.720,Gasoline
1857,2026-03-23,3.961,Gasoline
1666,2026-02-23,3.809,Diesel
1667,2026-03-02,3.897,Diesel
1668,2026-03-09,4.859,Diesel
1669,2026-03-16,5.071,Diesel
1670,2026-03-23,5.375,Diesel


In [169]:
# Find the peaks of gasolien and diesel prices
gas[gas['USA'] == gas['USA'].max()]





,date,USA,East Coast,New England (PADD 1A),Central Atlantic (PADD 1B),Lower Atlantic (PADD 1C),Midwest,Gulf Coast,Rocky Mountain,West Coast,...,Chicago,"Cleveland, OH",Denver,Houston,Los Angeles,"Miami, FL",New York City,San Francisco,"Seattle, WA",type
1660,2022-06-13,5.006,4.849,5.021,4.992,4.715,4.97,4.633,4.921,5.868,...,5.803,5.047,4.84,4.621,6.225,4.87,4.983,6.384,5.549,Gasoline


In [291]:
diesel[diesel['USA'] == diesel['USA'].max()]

,date,USA,East Coast,New England (PADD 1A),Central Atlantic (PADD 1B),Lower Atlantic (PADD 1C),Midwest,Gulf Coast,Rocky Mountain,West Coast,California,West Coast (PADD 5) Except California,type
1474,2022-06-20,5.81,5.883,6.123,6.129,5.762,5.78,5.453,5.782,6.516,6.915,6.151,Diesel


<br>
<br>
<br>

**Price Dispersion**

- East Coast (PADD1)
- New England (PADD1A)
- Central Atlantic (PADD1B)
- Lower Atlantic (PADD1C)
- Midwest (PADD2)
- Gulf Coast (PADD3)
- Rocky Mountain (PADD4)
- West Coast (PADD5)
- West Coast (less California)

In [170]:
pd.concat([gas[['date', 'USA', 'type']], diesel[['date', 'USA', 'type']]])

,date,USA,type
0,1990-08-20,1.191,Gasoline
1,1990-08-27,1.245,Gasoline
2,1990-09-03,1.242,Gasoline
3,1990-09-10,1.252,Gasoline
4,1990-09-17,1.266,Gasoline
...,...,...,...
1666,2026-02-23,3.809,Diesel
1667,2026-03-02,3.897,Diesel
1668,2026-03-09,4.859,Diesel
1669,2026-03-16,5.071,Diesel


In [126]:
diesel.columns

Index(['date', 'USA', 'East Coast', 'New England (PADD 1A)',
       'Central Atlantic (PADD 1B)', 'Lower Atlantic (PADD 1C)', 'Midwest',
       'Gulf Coast', 'Rocky Mountain', 'West Coast', 'California',
       'West Coast (PADD 5) Except California', 'type'],
      dtype='str')

In [ ]:
'Colorado', 'Florida', 'New York', 'Minnesota', 'Ohio', 'Texas', 'Washington', 'Cleveland, OH', 'Denver, CO', 'Miami, FL', 'Seattle, WA',

In [ ]:
gas.columns

Index(['date', 'USA', 'East Coast', 'New England (PADD 1A)',
       'Central Atlantic (PADD 1B)', 'Lower Atlantic (PADD 1C)', 'Midwest',
       'Gulf Coast', 'Rocky Mountain', 'West Coast', 'California', 'Colorado',
       'Florida', 'Massachusetts', 'Minnesota', 'New York', 'Ohio', 'Texas',
       'Washington', 'Boston, MA', 'Chicago', 'Cleveland, OH', 'Denver',
       'Houston', 'Los Angeles', 'Miami, FL', 'New York City', 'San Francisco',
       'Seattle, WA', 'type'],
      dtype='str')

In [296]:
gas

,date,USA,East Coast,New England (PADD 1A),Central Atlantic (PADD 1B),Lower Atlantic (PADD 1C),Midwest,Gulf Coast,Rocky Mountain,West Coast,...,Chicago,"Cleveland, OH",Denver,Houston,Los Angeles,"Miami, FL",New York City,San Francisco,"Seattle, WA",type
0,1990-08-20,1.191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gasoline
1,1990-08-27,1.245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gasoline
2,1990-09-03,1.242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gasoline
3,1990-09-10,1.252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gasoline
4,1990-09-17,1.266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gasoline
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1853,2026-02-23,2.937,2.834,2.852,2.962,2.748,2.675,2.532,2.662,4.111,...,2.909,2.816,2.556,2.483,4.407,2.866,2.855,4.566,4.389,Gasoline
1854,2026-03-02,3.015,2.882,2.878,2.967,2.830,2.794,2.644,2.758,4.160,...,3.028,2.859,2.842,2.610,4.460,2.882,2.865,4.590,4.427,Gasoline
1855,2026-03-09,3.502,3.363,3.352,3.419,3.330,3.276,3.109,3.258,4.690,...,3.555,3.412,3.433,3.044,5.081,3.495,3.321,5.235,4.688,Gasoline
1856,2026-03-16,3.720,3.577,3.538,3.612,3.566,3.393,3.412,3.637,4.987,...,3.613,3.443,3.848,3.344,5.411,3.720,3.564,5.483,4.894,Gasoline


In [297]:
diesel

,date,USA,East Coast,New England (PADD 1A),Central Atlantic (PADD 1B),Lower Atlantic (PADD 1C),Midwest,Gulf Coast,Rocky Mountain,West Coast,California,West Coast (PADD 5) Except California,type
0,1994-03-21,1.106,1.119,NaN,NaN,NaN,1.087,1.065,1.105,1.209,NaN,NaN,Diesel
1,1994-03-28,1.107,1.115,NaN,NaN,NaN,1.089,1.064,1.100,1.220,NaN,NaN,Diesel
2,1994-04-04,1.109,1.122,NaN,NaN,NaN,1.087,1.069,1.103,1.219,NaN,NaN,Diesel
3,1994-04-11,1.108,1.119,NaN,NaN,NaN,1.090,1.066,1.111,1.209,NaN,NaN,Diesel
4,1994-04-18,1.105,1.115,NaN,NaN,NaN,1.086,1.058,1.118,1.213,NaN,NaN,Diesel
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1666,2026-02-23,3.809,3.843,4.201,4.104,3.708,3.798,3.489,3.683,4.465,4.944,4.050,Diesel
1667,2026-03-02,3.897,3.924,4.314,4.122,3.808,3.888,3.598,3.737,4.534,4.990,4.138,Diesel
1668,2026-03-09,4.859,4.901,4.970,4.940,4.880,4.801,4.627,4.397,5.556,6.096,5.088,Diesel
1669,2026-03-16,5.071,5.105,5.236,5.196,5.057,4.970,4.835,4.796,5.856,6.428,5.360,Diesel


In [295]:
pd.melt(gas, id_vars=['date', 'type', 'USA'], var_name='region').groupby('region').tail(1).sort_values('value', ascending=False)

,date,type,USA,region,value
48307,2026-03-23,Gasoline,3.961,San Francisco,5.727
42733,2026-03-23,Gasoline,3.961,Los Angeles,5.668
16721,2026-03-23,Gasoline,3.961,California,5.650
50165,2026-03-23,Gasoline,3.961,"Seattle, WA",5.303
14863,2026-03-23,Gasoline,3.961,West Coast,5.262
31585,2026-03-23,Gasoline,3.961,Washington,5.160
35301,2026-03-23,Gasoline,3.961,Chicago,4.178
44591,2026-03-23,Gasoline,3.961,"Miami, FL",3.928
39017,2026-03-23,Gasoline,3.961,Denver,3.890
20437,2026-03-23,Gasoline,3.961,Florida,3.888


In [ ]:
['date', 'USA', 'East Coast', 'New England (PADD 1A)',
       'Central Atlantic (PADD 1B)', 'Lower Atlantic (PADD 1C)', 'Midwest',
       'Gulf Coast', 'Rocky Mountain', 'West Coast', 'Colorado', 'Florida',
       'New York', 'Minnesota', 'Ohio', 'Texas', 'Washington', 'Cleveland, OH',
       'Denver, CO', 'Miami, FL', 'Seattle, WA', 'type']

In [171]:
diesel.to_csv('usa_diesel_prices.csv', index=False)
gas.to_csv('usa_gasoline_prices.csv', index=False)

In [62]:
diesel_long = pd.melt(diesel.drop(columns=['California', 'West Coast (PADD 5) Except California']), id_vars=['date', 'type', 'USA'], var_name='region')


petrol_long = pd.melt(gas.drop(columns=['California', 'Colorado', 'Florida', 'New York', 'Massachusetts', 'Minnesota', 'New York City', 'Ohio', 'Texas', 'Washington', 'Boston, MA', 'Chicago', 'Cleveland, OH', 'Denver', 'Houston', 'Los Angeles', 'Miami, FL', 'New York', 'San Francisco', 'Seattle, WA']), id_vars=['date', 'type', 'USA'], var_name='region')


In [174]:
petrol_long['region'].unique()

<StringArray>
[                'East Coast',      'New England (PADD 1A)',
 'Central Atlantic (PADD 1B)',   'Lower Atlantic (PADD 1C)',
                    'Midwest',                 'Gulf Coast',
             'Rocky Mountain',                 'West Coast']
Length: 8, dtype: str

In [63]:
for df in [petrol_long, diesel_long]:
    df['min'] = df.groupby(['date'])['value'].transform('min')
    df['max'] = df.groupby(['date'])['value'].transform('max')
    df.dropna(subset=['value'], inplace=True)

petrol_long

,date,type,USA,region,value,min,max
90,1992-05-11,Gasoline,1.102,East Coast,1.071,1.071,1.189
91,1992-05-18,Gasoline,1.118,East Coast,1.085,1.085,1.208
92,1992-05-25,Gasoline,1.120,East Coast,1.092,1.092,1.218
93,1992-06-01,Gasoline,1.128,East Coast,1.100,1.100,1.228
94,1992-06-08,Gasoline,1.143,East Coast,1.114,1.114,1.246
...,...,...,...,...,...,...,...
14859,2026-02-23,Gasoline,2.937,West Coast,4.111,2.532,4.111
14860,2026-03-02,Gasoline,3.015,West Coast,4.160,2.644,4.160
14861,2026-03-09,Gasoline,3.502,West Coast,4.690,3.109,4.690
14862,2026-03-16,Gasoline,3.720,West Coast,4.987,3.393,4.987


In [176]:
styles.eco_palette()

{'Other_3': '#d6d4d4',
 'deemphasise_color': '#B4B4B4',
 'dark_grey': '#596870',
 'Deemphasise_Discrete': '#182a38',
 'Deemphasise_Continuous': '#182a3833',
 'accent': '#179fdb',
 'text': '#122b39',
 'domain': '#122b39',
 'non-uk': '#a8c0de',
 'country-group': '#eb5c2e',
 'background': '#fff'}

- Diesel: #00A767
- Gasoline: #122b39

In [177]:
petrol_long

,date,type,USA,region,value,min,max
90,1992-05-11,Gasoline,1.102,East Coast,1.071,1.071,1.189
91,1992-05-18,Gasoline,1.118,East Coast,1.085,1.085,1.208
92,1992-05-25,Gasoline,1.120,East Coast,1.092,1.092,1.218
93,1992-06-01,Gasoline,1.128,East Coast,1.100,1.100,1.228
94,1992-06-08,Gasoline,1.143,East Coast,1.114,1.114,1.246
...,...,...,...,...,...,...,...
14859,2026-02-23,Gasoline,2.937,West Coast,4.111,2.532,4.111
14860,2026-03-02,Gasoline,3.015,West Coast,4.160,2.644,4.160
14861,2026-03-09,Gasoline,3.502,West Coast,4.690,3.109,4.690
14862,2026-03-16,Gasoline,3.720,West Coast,4.987,3.393,4.987


In [291]:
petrol_long.groupby('region').tail(1)

,date,type,USA,region,value,min,max
1857,2026-03-23,Gasoline,3.961,East Coast,3.785,3.604,5.262
3715,2026-03-23,Gasoline,3.961,New England (PADD 1A),3.727,3.604,5.262
5573,2026-03-23,Gasoline,3.961,Central Atlantic (PADD 1B),3.855,3.604,5.262
7431,2026-03-23,Gasoline,3.961,Lower Atlantic (PADD 1C),3.756,3.604,5.262
9289,2026-03-23,Gasoline,3.961,Midwest,3.684,3.604,5.262
11147,2026-03-23,Gasoline,3.961,Gulf Coast,3.604,3.604,5.262
13005,2026-03-23,Gasoline,3.961,Rocky Mountain,3.850,3.604,5.262
14863,2026-03-23,Gasoline,3.961,West Coast,5.262,3.604,5.262


In [304]:
start_date = '2025-05-01'

data1 = petrol_long.drop_duplicates(subset=['date'])
data1 = data1[data1['date'] >= start_date].copy()

data2 = petrol_long[petrol_long['date'] >= start_date].copy()

spread = alt.Chart(data1).mark_area(color='#596870', opacity=0.15).encode(
    alt.X('date:T'),
    alt.Y('min:Q'),
    alt.Y2('max:Q')
)

usa = alt.Chart(data1).mark_line(strokeWidth=3).encode(
    alt.X('date:T'),
    alt.Y('USA:Q').scale(zero=False, domain=[1.6, 5.55], nice=False).axis(
        labelExpr="'$' + datum.value",
        labelFontSize=14, domain=False, tickCount=4,
        titleFontSize=12, gridDash=[2,2], labelPadding=6
    ).title('Regular gasoline, dollars per gallon'),
    color=alt.value('#122b39')
)
usa += usa.mark_text(align='left', dx=5, dy=0, text="US average - gasoline").encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('USA:Q').aggregate({'argmax': 'date'})
)

lines = alt.Chart(data2).mark_line(strokeWidth=1.5).transform_filter(
    # "datum.region == datum.min || datum.value == datum.max"
    "datum.region == 'Gulf Coast' || datum.region == 'West Coast'"
).encode(
    alt.X('date:T'),
    alt.Y('value:Q'),
    alt.Color('region:N').legend(None)
)
lines += lines.mark_text(align='left', dx=5, dy=0).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    alt.Text('region:N')
)

rule = alt.Chart(pd.DataFrame({'date': ['2026-02-28'], 'label': ['US & Israel |strikes on Iran']})).mark_rule(strokeWidth=1.5, opacity=0.5).encode(
    alt.X('date:T')
)
rule += rule.mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), y=alt.value(15)
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(-3),url='img:N')

chart = alt.layer(rule,spread, usa, lines, logo).properties(
    width=330, height=260
)



chart = styles.add_source(chart, 'Source: EIA; regular all formulations gasoline; price range across PADD areas.')

styles.save(chart, 'charts', f"usa-gasoline-prices-regional", width=330, height=260)
chart.display()


alt.LayerChart(...)

Notes:
- In latest datapoint, Midwest has overtaken Gulf Coast as PADD with lowest regular gasoline price (3.393 vs 3.412)
- California is even higher than West Coast (5.383 vs 4.987, or 4.521 for West Coast excluding Cali)
- US average 3.720

In [303]:
start_date = '2025-05-01'

data1 = diesel_long.drop_duplicates(subset=['date'])
data1 = data1[data1['date'] >= start_date].copy()

data2 = diesel_long[diesel_long['date'] >= start_date].copy()

spread = alt.Chart(data1).mark_area(color='#596870', opacity=0.15).encode(
    alt.X('date:T'),
    alt.Y('min:Q'),
    alt.Y2('max:Q')
)

usa = alt.Chart(data1).mark_line(strokeWidth=3).encode(
    alt.X('date:T'),
    alt.Y('USA:Q').scale(zero=False, domain=[2, 6.5], nice=False).axis(
        labelExpr="'$' + datum.value",
        labelFontSize=14, domain=False, tickCount=4,
        titleFontSize=12, gridDash=[2,2], labelPadding=6
    ).title('On-highway diesel, dollars per gallon'),
    color=alt.value('#122b39')
)
usa += usa.mark_text(align='left', dx=5, dy=0, text="US average - diesel").encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('USA:Q').aggregate({'argmax': 'date'})
)

lines = alt.Chart(data2).mark_line(strokeWidth=1.5).transform_filter(
    # "datum.region == datum.min || datum.value == datum.max"
    "datum.region == 'Gulf Coast' || datum.region == 'West Coast'"
).encode(
    alt.X('date:T'),
    alt.Y('value:Q'),
    alt.Color('region:N').legend(None)
)
lines += lines.mark_text(align='left', dx=5, dy=0).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'date'}),
    alt.Text('region:N')
)

rule = alt.Chart(pd.DataFrame({'date': ['2026-02-28'], 'label': ['US & Israel |strikes on Iran']})).mark_rule(strokeWidth=1.5, opacity=0.5).encode(
    alt.X('date:T')
)
rule += rule.mark_text(
    align='right', dx=-5, lineBreak="|", lineHeight=11
).encode(
    alt.Text('label:N'), y=alt.value(13)
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')

chart = alt.layer(rule, spread, usa, lines, logo).properties(
    width=330, height=260
)

chart = styles.add_source(chart, 'Source: EIA; on-highway no.2 diesel; price range across PADD areas.')

styles.save(chart, 'charts', f"usa-diesel-prices-regional", width=330, height=260)
chart.display()


alt.LayerChart(...)

In [302]:
from python import range

ModuleNotFoundError: No module named 'python'

In [230]:
diesel_long[diesel_long['value'] == diesel_long['min']].sort_values('date')

,date,type,USA,region,value,min,max
8350,1994-03-21,Diesel,1.106,Gulf Coast,1.065,1.065,1.209
8351,1994-03-28,Diesel,1.107,Gulf Coast,1.064,1.064,1.220
8352,1994-04-04,Diesel,1.109,Gulf Coast,1.069,1.069,1.219
8353,1994-04-11,Diesel,1.108,Gulf Coast,1.066,1.066,1.209
8354,1994-04-18,Diesel,1.105,Gulf Coast,1.058,1.058,1.213
...,...,...,...,...,...,...,...
10015,2026-02-16,Diesel,3.711,Gulf Coast,3.412,3.412,4.860
10016,2026-02-23,Diesel,3.809,Gulf Coast,3.489,3.489,4.944
10017,2026-03-02,Diesel,3.897,Gulf Coast,3.598,3.598,4.990
11688,2026-03-09,Diesel,4.859,Rocky Mountain,4.397,4.397,6.096


<br>
<br>

**4 week change in prices**

In [64]:
fuel = pd.read_csv('usa_average_petrol_prices.csv')

# Group by type, then calculate 4 week change in prices using a rolling window
fuel['4week_change'] = fuel.groupby('type')['USA'].transform(lambda x: x.diff(4))
fuel['4week_change_pct'] = fuel.groupby('type')['USA'].transform(lambda x: x.pct_change(4))
fuel




,date,USA,type,4week_change,4week_change_pct
0,1990-08-20,1.191,Gasoline,NaN,NaN
1,1990-08-27,1.245,Gasoline,NaN,NaN
2,1990-09-03,1.242,Gasoline,NaN,NaN
3,1990-09-10,1.252,Gasoline,NaN,NaN
4,1990-09-17,1.266,Gasoline,0.075,0.062972
...,...,...,...,...,...
3524,2026-02-23,3.809,Diesel,0.185,0.051049
3525,2026-03-02,3.897,Diesel,0.216,0.058680
3526,2026-03-09,4.859,Diesel,1.171,0.317516
3527,2026-03-16,5.071,Diesel,1.360,0.366478


In [70]:
# Find periods with high 4 week change in prices
fuel.sort_values('4week_change_pct', ascending=False).groupby('type').head(5)

,date,USA,type,4week_change,4week_change_pct
3528,2026-03-23,5.375,Diesel,1.566,0.411132
3527,2026-03-16,5.071,Diesel,1.360,0.366478
1857,2026-03-23,3.961,Gasoline,1.024,0.348655
3526,2026-03-09,4.859,Diesel,1.171,0.317516
3318,2022-03-14,5.250,Diesel,1.231,0.306295
785,2005-09-05,3.069,Gasoline,0.701,0.296030
1856,2026-03-16,3.720,Gasoline,0.796,0.272230
3319,2022-03-21,5.134,Diesel,1.079,0.266091
1647,2022-03-14,4.315,Gasoline,0.828,0.237453
980,2009-06-01,2.524,Gasoline,0.446,0.214629


In [239]:
# Filter to months around Hurricane Katrina
fuel[fuel['date'].between('2005-07-20', '2005-09-30')]
# Plot the 4 week change in prices

,date,USA,type,4week_change,4week_change_pct
779,2005-07-25,2.289,Gasoline,0.074,0.033409
780,2005-08-01,2.291,Gasoline,0.065,0.029200
781,2005-08-08,2.368,Gasoline,0.040,0.017182
782,2005-08-15,2.550,Gasoline,0.233,0.100561
783,2005-08-22,2.612,Gasoline,0.323,0.141110
784,2005-08-29,2.610,Gasoline,0.319,0.139241
785,2005-09-05,3.069,Gasoline,0.701,0.296030
786,2005-09-12,2.955,Gasoline,0.405,0.158824
787,2005-09-19,2.786,Gasoline,0.174,0.066616
788,2005-09-26,2.803,Gasoline,0.193,0.073946


In [290]:
fuel[fuel['date'].between('2022-02-20', '2022-03-30')]

,date,USA,type,4week_change,4week_change_pct
1644,2022-02-21,3.530,Gasoline,0.207,0.062293
1645,2022-02-28,3.608,Gasoline,0.240,0.071259
1646,2022-03-07,4.102,Gasoline,0.658,0.191057
1647,2022-03-14,4.315,Gasoline,0.828,0.237453
1648,2022-03-21,4.239,Gasoline,0.709,0.200850
1649,2022-03-28,4.231,Gasoline,0.623,0.172672
3315,2022-02-21,4.055,Diesel,0.275,0.072751
3316,2022-02-28,4.104,Diesel,0.258,0.067083
3317,2022-03-07,4.849,Diesel,0.898,0.227284
3318,2022-03-14,5.250,Diesel,1.231,0.306295


In [289]:
fuel.groupby('type').tail(5)

,date,USA,type,4week_change,4week_change_pct
1853,2026-02-23,2.937,Gasoline,0.084,0.029443
1854,2026-03-02,3.015,Gasoline,0.148,0.051622
1855,2026-03-09,3.502,Gasoline,0.600,0.206754
1856,2026-03-16,3.720,Gasoline,0.796,0.272230
1857,2026-03-23,3.961,Gasoline,1.024,0.348655
3524,2026-02-23,3.809,Diesel,0.185,0.051049
3525,2026-03-02,3.897,Diesel,0.216,0.058680
3526,2026-03-09,4.859,Diesel,1.171,0.317516
3527,2026-03-16,5.071,Diesel,1.360,0.366478
3528,2026-03-23,5.375,Diesel,1.566,0.411132


- red/pink: #E6224B
- orange: #EB5C2E

In [72]:
bars = alt.Chart(fuel).mark_bar(width=2.3, opacity=1).transform_filter(
    "year(datum.date) >= 2024"
).transform_joinaggregate(
    max_date='max(date)',  # latest date within each 'type'
    groupby=['type']
).encode(
    alt.X('date:T').axis(tickSize=10, tickCount=4, domain=False),
    alt.Y('4week_change_pct:Q').axis(
        gridDash=alt.expr('datum.value == 0 ? [0,0] : [1,5]'),
        gridWidth=alt.expr('datum.value == 0 ? 2.2 : 1'),
        labelExpr="datum.value > 0 ? '+' + datum.value * 100 + '%' : datum.value * 100 + '%'",
        tickCount=5,
        titleY=12,
        titleX=3, titleLineHeight=10
    ).title(['4-week price change']),
        # TODO: conditionally colour bar if it is the last (i.e. most recent value) in the series. E.g. argmax for latest value on date
    color=alt.condition(
        'time(datum.date) === time(datum.max_date)',  # only the most recent bar
        alt.value('#E6224B'),
        alt.value('#676A86')
    )
).properties(
    width=330,
    height=200
)

bars += bars.mark_text(
    align='right',
    dx=-5,
    dy=-1,
    fontStyle='bold',
    fontSize=13
).encode(
    alt.X('date:T').aggregate('max'),
    alt.Y('4week_change_pct:Q').aggregate({'argmax': 'date'}),
    alt.Text('4week_change_pct:N').aggregate({'argmax': 'date'}).format('+.1%'),
    color=alt.value('#E6224B')
)

chart = bars.facet(row=alt.Row('type').title('').header(labelAngle=0, labelAnchor='middle', labelAlign='center', orient='top', labelFont='Circular Std', labelFontWeight='bold', labelFontSize=13, labelPadding=-27, labelColor='#676A86'))

chart.title = alt.Title(
    text='Source: EIA; regular all forms gasoline & No.2 diesel, weekly retail prices',
    align='left',
    fontStyle='italic',
    fontSize=10,
    color='#676A8680',  # domain colour with 50% opacity
    orient='bottom',
    frame='group',
    offset=10
)
chart = chart.configure_axis(labelFontSize=12)

styles.save(chart, 'charts', f"usa-fuel-prices-4week-change", width=None, height=None)
chart.display()

alt.FacetChart(...)

In [ ]:
# Plot the 4 week change in prices
fuel.plot(x='date', y='4week_change', kind='line', hue='type')
plt.show()

<br>
<br>
<br>

### Comparing shocks

Prep data

In [ ]:
usa_average_petrol_prices.csv()

In [74]:
"""
gasoline_shock_index.py
Builds an indexed comparison chart of major US gasoline price shocks since 1990.
Data: EIA Weekly Regular All Formulations Retail Gasoline Prices ($/gallon)

Week 0 = first available EIA weekly reading at or after each event date.
Index: week 0 price = 100.
"""

import pandas as pd
import altair as alt
import numpy as np

# ── Config ──────────────────────────────────────────────────────────────────

DATA_PATH = "usa_average_petrol_prices.csv"

WEEKS = 18  # how many weeks to show for each event

# Event dates — week 0 will be the last EIA reading ON OR BEFORE this date.
# Gulf War uses a synthetic pre-invasion date derived from FRED CPI.
EVENTS = {
    "Gulf War|(1990)":           "1990-07-30",   # Iraq invades Kuwait 2 Aug; nearest reading
    # "Hurricane Katrina (2005)":  "2005-08-29",   # landfall; spike visible in Sep 5 reading
    "Hurricane Katrina|(2005)":  "2005-08-23",   # formed over bahamas;
    # "2008 Oil Surge":            "2007-12-31",   # slow demand-driven run-up Jan→Jul 2008
    "Ukraine Invasion|(2022)":   "2022-02-24",   # invasion Feb 24; nearest reading
    "Iran War|(2026)":           "2026-02-28",   # US-Israel strikes Feb 28; nearest reading
}

# Colour palette — Iran in ECO red, Ukraine in dark navy, others in softer tones
PALETTE = {
    "Gulf War|(1990)":           "#8B9EB7",
    "Hurricane Katrina|(2005)":  "#E8A838",
    "2008 Oil Surge":            "#6BAF92",
    "Ukraine Invasion|(2022)":   "#122b39",
    "Iran War|(2026)":           "#e6224b",
}

# ── 1. Load & prep series ────────────────────────────────────────────────────

avg = pd.read_csv(DATA_PATH)
avg["date"] = pd.to_datetime(avg["date"])
gas = avg[avg["type"] == "Gasoline"].set_index("date")["USA"].sort_index()


# ── 2. Gulf War synthetic pre-invasion price ─────────────────────────────────
#
# EIA data begins 1990-08-20 — 18 days after Iraq invaded Kuwait on Aug 2.
# By that point prices had already risen ~8%. To capture the full shock we
# prepend a synthetic week-0 price (Jul 30) and two interpolated bridge weeks
# (Aug 6, Aug 13) before the EIA series takes over at Aug 20.
#
# Method: use FRED CPI Gasoline (CUSR0000SETB01) monthly values to derive the
# pre-invasion level, then anchor to the first EIA reading via the CPI ratio.
#   Jul 1990 CPI = 93.1  (clean pre-invasion month)
#   Aug 1990 CPI = 100.9 (covers the invasion and immediate spike)
#   First EIA reading: Aug 20 = $1.191
#
# Pre-invasion price = EIA_aug20 * (CPI_jul / CPI_aug) = 1.191 * (93.1/100.9)
 
CPI_JUL_1990  = 93.1
CPI_AUG_1990  = 100.9
EIA_AUG20     = 1.191   # first EIA reading
EIA_AUG27     = 1.245   # second EIA reading

gw_w0_price = EIA_AUG20 * (CPI_JUL_1990 / CPI_AUG_1990)   # approx $1.099
 
# Synthetic dates and prices for the pre-EIA gap weeks
# Linearly interpolate prices: Jul 30 -> Aug 20 (3 steps over 3 weeks)
gw_synthetic = pd.Series({
    pd.Timestamp("1990-07-30"): gw_w0_price,
    pd.Timestamp("1990-08-06"): gw_w0_price + (EIA_AUG20 - gw_w0_price) * 1/3,
    pd.Timestamp("1990-08-13"): gw_w0_price + (EIA_AUG20 - gw_w0_price) * 2/3,
})
 
# Splice: synthetic weeks + EIA from Aug 20 onward
gw_eia = gas.loc["1990-08-20":]
gulf_war_series = pd.concat([gw_synthetic, gw_eia]).sort_index()


# ── 3. Helper: snap to last EIA reading on/before event date ─────────────────
 
def get_w0_idx(series: pd.Series, event_date: str) -> int:
    """Return integer position of last observation on or before event_date."""
    ts   = pd.Timestamp(event_date)
    mask = series.index <= ts
    if not mask.any():
        raise ValueError(f"No data on or before {event_date}")
    return int(np.where(mask)[0][-1])


# ── 4. Build indexed dataframe ───────────────────────────────────────────────
 
records = []
 
for label, event_date in EVENTS.items():
 
    # Use the augmented Gulf War series; all others use the standard EIA series
    series = gulf_war_series if label == "Gulf War|(1990)" else gas
 
    idx0       = get_w0_idx(series, event_date)
    base_price = series.iloc[idx0]
    w0_date    = series.index[idx0]
 
    for week in range(WEEKS + 1):
        idx = idx0 + week
        if idx >= len(series):
            break
        price = series.iloc[idx]
        if pd.isna(price):          # skip EIA coverage gaps (e.g. Dec 1990)
            continue
        records.append({
            "event":        label,
            "week":         week,
            "price":        round(price, 3),
            "index":        round(price / base_price * 100, 2),
            "base_price":   round(base_price, 3),
            "w0_date":      w0_date.strftime("%Y-%m-%d"),
            "is_current":   label == "Iran War|(2026)",
            "is_synthetic": label == "Gulf War|(1990)" and week <= 2,
        })
 
df = pd.DataFrame(records)

In [75]:
# ── 5. Summary ───────────────────────────────────────────────────────────────
 
def summary_row(g):
    peak_row = g.loc[g["index"].idxmax()]
    return pd.Series({
        "week_0_date":   g.iloc[0]["w0_date"],
        "week_0_price":  g.iloc[0]["base_price"],
        "peak_index":    peak_row["index"],
        "peak_week":     peak_row["week"],
        "final_index":   g.iloc[-1]["index"],
        "weeks_of_data": g["week"].max(),
    })
 
summary = df.groupby("event", sort=False).apply(summary_row, include_groups=False)
summary = summary.loc[list(EVENTS.keys())]
print(summary.to_string())
print()

                         week_0_date  week_0_price  peak_index  peak_week  final_index  weeks_of_data
event                                                                                                
Gulf War|(1990)           1990-07-30         1.099      122.39         12       122.03             18
Hurricane Katrina|(2005)  2005-08-22         2.612      117.50          2        84.11             18
Ukraine Invasion|(2022)   2022-02-21         3.530      141.81         16       138.02             18
Iran War|(2026)           2026-02-23         2.937      134.87          4       134.87              4



Choosing dates:
- Aug 22nd best baseline for Hurricane Katrina:
    - Hurricane Katrina did not form as a tropical depression until 23 August [1.1]. The week ending 22nd August represents the final period of market activity before the storm existed and began influencing trader sentiment or supply logistics.
    - Separation from Pre-Existing Rises: You noted a price increase between 1 August and 15 August. This earlier rise was driven by global supply-demand imbalances and record-high crude oil prices (which hit nearly $70/barrel for the first time in mid-August), rather than the hurricane itself.

In [77]:
# Ordered for legend / colour scale
event_order  = list(EVENTS.keys())
color_range  = [PALETTE[e] for e in event_order]

# ── 3. Chart components ──────────────────────────────────────────────────────

shared_color = alt.Color("event:N").scale(
    domain=event_order, range=color_range
).legend(None)

# Baseline rule at index = 100
rule = alt.Chart(
    pd.DataFrame({"y": [100]})
).mark_rule(
    color="#676A86",
    strokeDash=[4, 3],
    strokeWidth=1,
).encode(y="y:Q")

# Main lines
lines = alt.Chart(df).mark_line(interpolate="monotone").encode(
    x=alt.X("week:Q").title("Weeks since event").scale(domain=[0, WEEKS]).axis(
        tickCount=WEEKS // 2,
        grid=True,
        gridOpacity=0.12,
        tickMinStep=1,
        titleY=-14
    ),
    y=alt.Y("index:Q").title("Price index (week 0 = 100)").scale(zero=False, domain=[70, 145]).axis(
        grid=True,
        gridOpacity=0.12,
        tickCount=7
    ),
    color=shared_color,
    strokeWidth=alt.condition(
        alt.datum.is_current,
        alt.value(3),
        alt.value(2),
    ),
    opacity=alt.condition(
        alt.datum.is_current,
        alt.value(1.0),
        alt.value(0.80),
    ),
    tooltip=[
        alt.Tooltip("event:N",   title="Event"),
        alt.Tooltip("week:Q",    title="Week"),
        alt.Tooltip("index:Q",   title="Index", format=".1f"),
        alt.Tooltip("price:Q",   title="$/gal",  format=".3f"),
    ],
)

labels = alt.Chart(df).mark_text(
    align='left', dx=6, dy=0, lineBreak='|', 
).encode(
    alt.X('week:Q').aggregate('max'),
    alt.Y('index:Q').aggregate({'argmax': 'week'}),
    alt.Text('event:N').aggregate({'argmax': 'week'}),
    color=shared_color
)

# Dot at the final data point for Iran (signals series is incomplete)
iran_tip = df[df["is_current"] & (df["week"] == df.loc[df["is_current"], "week"].max())]
dot = alt.Chart(iran_tip).mark_point(filled=True, size=70).encode(
    x="week:Q",
    y="index:Q",
    color=alt.Color(
        "event:N",
        scale=alt.Scale(domain=event_order, range=color_range),
        legend=None,
    ),
)

# ── 4. Assemble ───────────────────────────────────────────────────────────────

source = alt.Chart(pd.DataFrame({'x': [0, 1, 2]})).mark_text(
    text=[
        'Source: EIA; pre 1990-08-20 data for Gulf War derived from FRED CPI',
        'Regular all formulations retail price. Week 0 = 100 for each event.'
    ],
    fontSize=10,
    opacity=0.5,
    dx=0,
    dy=30,
    align='left'
).encode(
    x=alt.value(0),
    y=alt.value('height')
)


chart = (rule + lines + labels + dot + source + logo).properties(
    width=320,
    height=350,
    title=alt.Title(
        "US gasoline price shocks: comparison",
        subtitleColor="#676A8680",
        subtitleFontSize=10.5,
        subtitleLineHeight=14,
        offset=0
    ),
).configure_view(
    strokeWidth=0,
).configure_axis(
    labelFontSize=11,
    titleFontSize=11,
).configure_title(
    fontSize=13,
    fontWeight="bold",
    anchor="start",
)

# ── 5. Export ─────────────────────────────────────────────────────────────────

# chart.save("gasoline_shock_index.html")
# df.to_csv("gasoline_shock_index_data.csv", index=False)
# print("Saved: gasoline_shock_index.html, gasoline_shock_index_data.csv")

# ── 6. Quick summary ──────────────────────────────────────────────────────────

summary = (
    df.groupby("event")
    .apply(lambda g: pd.Series({
        "week_0_price":  g.loc[g["week"] == 0, "price"].values[0],
        "peak_index":    g["index"].max(),
        "peak_week":     g.loc[g["index"].idxmax(), "week"],
        "final_index":   g.loc[g["week"].idxmax(), "index"],
        "weeks_of_data": g["week"].max(),
    }), include_groups=False)
    .loc[event_order]
)
print("\n" + summary.to_string())

styles.save(chart, 'charts', f"gas-price-shocks-comparison", width=320, height=350)
chart.display()


                          week_0_price  peak_index  peak_week  final_index  weeks_of_data
event                                                                                    
Gulf War|(1990)                  1.099      122.39       12.0       122.03           18.0
Hurricane Katrina|(2005)         2.612      117.50        2.0        84.11           18.0
Ukraine Invasion|(2022)          3.530      141.81       16.0       138.02           18.0
Iran War|(2026)                  2.937      134.87        4.0       134.87            4.0


alt.LayerChart(...)

In [5]:
chart

alt.LayerChart(...)